# SynBioCrow 2.4.10 — Exact Galaxy target-blacklist recovery\nRecovers Galaxy-SynBioCAD Supplementary Dataset 2, reconstructs the exact frozen 65-path target blacklist using the released SynBioCrow 2.3 split policy, resolves the 2.4 source tranche, persists results to Google Drive, and fails closed on count/schema mismatches.\n

In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile, requests

drive.mount("/content/drive", force_remount=False)

REPO=Path("/content/SynBioCrow_2_4_10")
OUT=Path("/content/drive/MyDrive/SynBioCrow/2.4/target_blacklist_2_4_10")
OUT.mkdir(parents=True,exist_ok=True)
XLSX=Path("/content/Galaxy_SynBioCAD_Supplementary_Dataset_2.xlsx")

def run(cmd,*,cwd=None,timeout=1800):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)
    return p

print("[2.4.10]  5% | clone develop/2.4",flush=True)
if REPO.exists(): shutil.rmtree(REPO)
run(["git","clone","--depth","1","--branch","develop/2.4","https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)
run([sys.executable,"-m","pip","install","-q","rdkit","openpyxl","requests"],timeout=900)

print("[2.4.10] 20% | recover Galaxy Supplementary Dataset 2",flush=True)
urls=[
 "https://media.springernature.com/original/springer-static/esm/art%3A10.1038%2Fs41467-022-32661-x/MediaObjects/41467_2022_32661_MOESM6_ESM.xlsx",
 "https://media.springernature.com/original/springer-static/esm/art:10.1038/s41467-022-32661-x/MediaObjects/41467_2022_32661_MOESM6_ESM.xlsx",
]
ok=False
for u in urls:
    try:
        r=requests.get(u,timeout=60)
        print("  URL",r.status_code,len(r.content),u,flush=True)
        if r.ok and len(r.content)>30000 and r.content[:2]==b"PK":
            XLSX.write_bytes(r.content); ok=True; break
    except Exception as e:
        print("  download failed:",type(e).__name__,e,flush=True)

if not ok:
    print("\nAutomatic publisher download unavailable.",flush=True)
    print("Upload ONLY Galaxy-SynBioCAD Supplementary Dataset 2 XLSX (about 55 KB).",flush=True)
    up=files.upload()
    xs=[Path("/content")/n for n in up if n.lower().endswith(".xlsx")]
    assert xs,"No XLSX uploaded"
    shutil.copy2(xs[0],XLSX)

print("[2.4.10] 35% | source bytes",XLSX.stat().st_size,"sha256",hashlib.sha256(XLSX.read_bytes()).hexdigest(),flush=True)

TRANCHE=REPO/"benchmarks/2.4/source_tranche_v1.json"
RUNOUT=Path("/content/synbiocrow_2_4_10_target_blacklist")
if RUNOUT.exists(): shutil.rmtree(RUNOUT)

print("[2.4.10] 45% | reconstruct 77-path source and exact 65-path holdout",flush=True)
run([sys.executable,str(REPO/"scripts/v24_recover_target_blacklist.py"),
     "--dataset2-xlsx",str(XLSX),
     "--tranche",str(TRANCHE),
     "--out-dir",str(RUNOUT)],cwd=REPO,timeout=900)

summary=json.loads((RUNOUT/"recovery_summary.json").read_text())
assert summary["heldout_pathway_count"]==65
assert summary["pending_count"]==0, summary
print("[2.4.10] 75% | exact blacklist recovered; tranche fully resolved",flush=True)
print(json.dumps(summary,indent=2),flush=True)

for p in RUNOUT.iterdir():
    shutil.copy2(p,OUT/p.name)

bundle=OUT/"SynBioCrow_2_4_10_TARGET_BLACKLIST_RECOVERY_ALL_OUTPUTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and p!=bundle:
            z.write(p,p.name)

print("[2.4.10] 95% | persisted",bundle,flush=True)
print("[2.4.10] SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest(),flush=True)
print("[2.4.10] 100% | PASS",flush=True)
files.download(str(bundle))


## Rescue download\nRun this only if the automatic browser download was missed.\n

In [ ]:
from google.colab import files
from pathlib import Path
p=Path("/content/drive/MyDrive/SynBioCrow/2.4/target_blacklist_2_4_10/SynBioCrow_2_4_10_TARGET_BLACKLIST_RECOVERY_ALL_OUTPUTS.zip")
assert p.is_file(), p
files.download(str(p))
